In [1]:
# imports
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# PT3 imports
from distfit import distfit                          # fits statistical distributions to data
from scipy.stats import pearson3, norm              # PT3 distribution and normal distribution tools

# sci-kit learn imports
from sklearn.linear_model import ElasticNet         # the model we're training
from sklearn.model_selection import GridSearchCV    # cross-validated hyperparameter search
from sklearn.metrics import mean_squared_error      # computes MSE
from sklearn.metrics import r2_score

# for data transfer
import pickle


In [2]:
# load data

with open('../data/processed/kmeans.pkl', 'rb') as f:
    data = pickle.load(f)

X_train_clustered = data['X_train_clustered']
X_train_scaled = data['X_train_scaled'].drop(columns=['Cluster'], errors='ignore')
X_test_scaled = data['X_test_scaled']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
k_means_labels = data['K_means_labels']


In [3]:
# show data

X_train_scaled.head()

,HSBASHHD,HSFD001S,HSHO001S,HSHF001S,HSTR001S,HSHC001S,HSPC001S,HSRE001S,HSRO001S,HSTA018S,...,ECYGEN1GEN,ECYGEN2GEN,ECYGEN3GEN,ECYTCAHPOP,ECYTCACIT,ECYTCA_U18,ECYTCA_18P,ECYNCANCIT,ECYNCA_U18,ECYNCA_18P
0,2.096885,1.924531,1.521558,1.919040,1.829394,2.207040,1.950844,2.051773,0.913626,1.965526,...,-0.228224,-0.121147,4.314653,1.727888,2.247607,2.242034,2.181706,-0.248492,-0.197283,-0.255273
1,-0.332897,-0.416294,-0.376221,-0.305056,0.000908,-0.388333,-0.318336,-0.293081,-0.566061,-0.265945,...,-0.520022,-0.405742,0.054563,-0.397312,-0.345664,-0.138669,-0.404826,-0.415133,-0.352709,-0.421172
2,-0.368225,-0.380159,-0.381707,-0.348902,-0.490942,-0.514085,-0.318584,-0.254495,0.029062,-0.317163,...,-0.375272,-0.179227,-0.136139,-0.336145,-0.329678,-0.200775,-0.363070,-0.248492,-0.274996,-0.237498
3,0.511050,0.525045,0.682765,0.458800,0.522289,0.285856,0.419810,0.668126,-0.139953,0.733696,...,-0.078878,-0.051450,1.383013,0.547221,0.645463,0.875717,0.548595,0.104396,0.139473,0.094301
4,0.063562,-0.177893,-0.182823,-0.271947,-0.167056,-0.100117,-0.179068,-0.200501,-0.232490,-0.145046,...,-0.225926,-0.173419,0.006080,-0.179671,-0.203567,-0.359488,-0.145013,-0.057345,-0.171379,-0.030124


In [4]:
# ============================================================
# STEP 3a-i — Fit PT3 distribution and transform y → z_normal
# ============================================================

# --- Fit PT3 to y_train ---

# distfit tries many distributions; we restrict it to just 'pearson3'
# so it finds the best-fitting PT3 parameters on the training set only.
dfit = distfit(distr='pearson3')
dfit.fit_transform(y_train.values if hasattr(y_train, 'values') else y_train)

# Extract the three PT3 parameters: skewness (skew), location (loc), scale (scale).
# These describe the shape of your data's distribution.
pt3_params = dfit.model['params']   # returns (skew, loc, scale)
skew, loc, scale = pt3_params

print(f"PT3 params — skew: {skew:.4f}, loc: {loc:.4f}, scale: {scale:.4f}")


# --- Helper: transform y → z_normal ---

def transform_to_znormal(y, skew, loc, scale, eps=1e-6):
    """
    Converts raw y values into z_normal:
      1. Evaluate the PT3 CDF at each y value → gives a probability in [0,1].
      2. Clip away from 0 and 1 to prevent the next step returning ±infinity.
      3. Apply the inverse standard-normal CDF (PPF) → gives z values ~ N(0,1).
    """
    y_arr = np.array(y)

    # Step 1: PT3 CDF — "what percentile is each value?"
    cdf_vals = pearson3.cdf(y_arr, skew, loc=loc, scale=scale)

    # Step 2: Clip to (eps, 1-eps) so norm.ppf never hits -inf or +inf
    cdf_vals = np.clip(cdf_vals, eps, 1 - eps)

    # Step 3: Normal PPF — "what z-score corresponds to this percentile?"
    z = norm.ppf(cdf_vals)
    return z


# --- Transform training and test targets ---

# We fit PT3 only on y_train; we apply the same parameters to y_test
# (no "peeking" at the test distribution).
z_train = transform_to_znormal(y_train, skew, loc, scale)
z_test  = transform_to_znormal(y_test,  skew, loc, scale)

print(f"z_train — mean: {z_train.mean():.3f}, std: {z_train.std():.3f}")

[09-04-2026 16:06:25] [distfit.distfit] [INFO] fit
[09-04-2026 16:06:25] [distfit.distfit] [INFO] transform
[09-04-2026 16:06:26] [distfit.distfit] [INFO] [pearson3] [0.65 sec] [RSS: 5.21034] [loc=0.257 scale=0.049]
[09-04-2026 16:06:26] [distfit.distfit] [INFO] [pearson3] [0.65 sec] [RSS: 5.21034] [loc=0.257 scale=0.049]
[09-04-2026 16:06:26] [distfit.distfit] [INFO] Compute confidence intervals [parametric]


PT3 params — skew: 0.9501, loc: 0.2570, scale: 0.0489
z_train — mean: -0.004, std: 0.984


In [5]:
# ============================================================
# STEP 3a-iii — Train ElasticNet via GridSearchCV
# ============================================================

# Define the hyperparameter grid.
# alpha controls overall regularisation strength (larger = stronger penalty).
# l1_ratio controls the mix: 0 = Ridge (L2 only), 1 = Lasso (L1 only).
param_grid = {
    'alpha':     [1e-2, 5e-2, 1e-1, 5e-1, 1.0],   # 5 alpha values spanning [1e-4, 1]
    'l1_ratio':  [0.1, 0.25, 0.5, 0.75, 1.0],      # 5 l1_ratio values in [0, 1]
}

# Instantiate the base model.
# max_iter is raised so the solver has enough steps to converge on all folds.
en = ElasticNet(
    max_iter=20_000,
    tol=1e-3,              # loosened from default 1e-4
)

# GridSearchCV wraps the model and tries every (alpha, l1_ratio) combination
# using 5-fold cross-validation, scoring by R² (default for regressors).
grid_search = GridSearchCV(
    estimator=en,
    param_grid=param_grid,
    cv=5,                  # 5-fold CV as required
    scoring='r2',
    n_jobs=-1,             # use all CPU cores
    verbose=1
)

# Fit on the scaled training features and the z_normal-transformed target.
grid_search.fit(X_train_scaled, z_train)

best_model = grid_search.best_estimator_   # the model with best CV score
best_params = grid_search.best_params_
print(f"Best params: {best_params}")
print(f"Best CV R² (on z_train): {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 3a-iv — Reverse-transform predictions back to y space
# ============================================================

# --- Helper: reverse z_normal → y ---

def inverse_transform(z_pred, skew, loc, scale, eps=1e-6):
    """
    Converts z_normal predictions back to the original y scale:
      1. Apply the normal CDF to z → gets back a probability in [0,1].
      2. Clip to avoid edge issues (mirroring what we did during forward transform).
      3. Apply the PT3 PPF (inverse CDF) → recovers the original y scale.
    """
    # Step 1: Normal CDF — "what percentile is each z-score?"
    p = norm.cdf(z_pred)

    # Step 2: Clip for numerical safety
    p = np.clip(p, eps, 1 - eps)

    # Step 3: PT3 PPF — "what y value sits at this percentile?"
    y_pred = pearson3.ppf(p, skew, loc=loc, scale=scale)
    return y_pred


# Predict z on the test set, then reverse-transform to original y scale.
z_pred_test = best_model.predict(X_test_scaled)
y_pred       = inverse_transform(z_pred_test, skew, loc, scale)

# Also get training predictions (useful for diagnostics).
z_pred_train = best_model.predict(X_train_scaled)
y_pred_train  = inverse_transform(z_pred_train, skew, loc, scale)

In [ ]:
# ============================================================
# STEP 3a-v — Report metrics and scatterplot
# ============================================================

# --- Point estimates ---

r2_test  = r2_score(y_test, y_pred)
mse_test = mean_squared_error(y_test, y_pred)

print(f"\nTest R²:  {r2_test:.4f}")
print(f"Test MSE: {mse_test:.4f}")

# --- Bootstrapped 95% CI for R² (1,000 iterations) ---

np.random.seed(42)
n_boot = 1_000                            # number of bootstrap resamples
boot_r2 = np.empty(n_boot)               # will store one R² per resample

y_test_arr = np.array(y_test)            # make sure we can index by integer array
y_pred_arr = np.array(y_pred)

for i in range(n_boot):
    # Draw n indices with replacement from the test set.
    idx = np.random.randint(0, len(y_test_arr), size=len(y_test_arr))
    boot_r2[i] = r2_score(y_test_arr[idx], y_pred_arr[idx])

# The 2.5th and 97.5th percentiles of the bootstrap distribution form the 95% CI.
ci_low, ci_high = np.percentile(boot_r2, [2.5, 97.5])

print(f"\nBootstrapped 95% CI for R²: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Bootstrap mean R²:          {boot_r2.mean():.4f}")

# --- Scatterplot: predicted vs actual ---

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test, y_pred, alpha=0.4, s=20, color='steelblue', label='Test samples')

# Perfect-prediction reference line (y = x)
lims = [min(y_test_arr.min(), y_pred_arr.min()),
        max(y_test_arr.max(), y_pred_arr.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction (y = x)')

ax.set_xlabel('Actual y')
ax.set_ylabel('Predicted ŷ')
ax.set_title(
    f'Predicted vs Actual  |  R² = {r2_test:.3f}  |  95% CI [{ci_low:.3f}, {ci_high:.3f}]'
)
ax.legend()
plt.tight_layout()
plt.show()

# --- Interpretation notes (printed for reference) ---
print("""
Scatterplot interpretation:
  - Points clustered tightly around the red dashed line → good predictions.
  - Systematic curves (fan shape, banana) → remaining non-linearity or
    heteroskedasticity not captured by the linear model.
  - A wide CI means the R² estimate is unstable — possibly due to small test set.
""")

In [ ]:
# ============================================================
# STEP 3a-vi — Per K-Means group R²
# ============================================================

# 'kmeans_labels_test' should be a 1-D array/Series of integer cluster IDs
# aligned with X_test and y_test. Rename if yours is called something different.
# e.g. if you stored them as 'labels_test', replace the name below.

kmeans_labels_test = np.array(k_means_labels)   # ensure it is a numpy array

unique_labels = np.unique(k_means_labels)
print("\nPer-group R² (K-Means clusters on test set):")
print("-" * 40)

group_r2 = {}   # store results for comparison

for label in unique_labels:
    mask = k_means_labels == label          # boolean mask for this cluster
    r2_group = r2_score(y_test_arr[mask], y_pred_arr[mask])
    group_r2[label] = r2_group
    n_group = mask.sum()
    print(f"  Cluster {label}  (n={n_group:4d})  R² = {r2_group:.4f}")

print("-" * 40)
print(f"  Overall test R²:          {r2_test:.4f}")

# --- Quick bar chart for visual comparison ---
fig2, ax2 = plt.subplots(figsize=(7, 4))
labels_sorted = sorted(group_r2.keys())
r2_vals = [group_r2[k] for k in labels_sorted]

ax2.bar([f'Cluster {k}' for k in labels_sorted], r2_vals,
        color='steelblue', alpha=0.75, edgecolor='white')
ax2.axhline(r2_test, color='red', linestyle='--', linewidth=1, label=f'Overall R² = {r2_test:.3f}')
ax2.set_ylabel('R²')
ax2.set_title('R² by K-Means cluster (test set)')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 3a-vii — Non-zero coefficients and top 5 by magnitude
# ============================================================

# Retrieve the coefficients from the best ElasticNet model.
# Each coefficient corresponds to one input feature, telling us
# how much a one-unit change in that feature shifts z_normal (our transformed target).
coefficients = best_model.coef_

# Get the feature names from our scaled training DataFrame.
feature_names = X_train_scaled.columns if hasattr(X_train_scaled, 'columns') else [f'feature_{i}' for i in range(len(coefficients))]

# Build a Series pairing each feature name with its coefficient.
coef_series = pd.Series(coefficients, index=feature_names)

# --- Non-zero coefficients ---
# ElasticNet's L1 penalty drives less useful coefficients exactly to zero.
# The ones that survive (non-zero) are the features the model found meaningful.
nonzero_coefs = coef_series[coef_series != 0]
print(f"Total features:            {len(coef_series)}")
print(f"Non-zero coefficients:     {len(nonzero_coefs)}")
print(f"Zeroed-out (L1 excluded):  {len(coef_series) - len(nonzero_coefs)}")

# --- Top 5 by magnitude ---
# We sort by absolute value because a large negative coefficient is just as
# influential as a large positive one — both indicate strong relationships.
top5 = coef_series.reindex(coef_series.abs().sort_values(ascending=False).index).head(5)

print("\nTop 5 coefficients by magnitude:")
print("-" * 45)
for feat, val in top5.items():
    direction = "↑ positive" if val > 0 else "↓ negative"
    print(f"  {feat:<30} {val:+.4f}  ({direction})")
print("-" * 45)

# --- Bar chart of top 5 ---
fig, ax = plt.subplots(figsize=(8, 4))

colors = ['steelblue' if v > 0 else 'coral' for v in top5.values]
ax.barh(top5.index[::-1], top5.values[::-1], color=colors[::-1], edgecolor='white')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient value (in z_normal space)')
ax.set_title('Top 5 ElasticNet coefficients by magnitude')
plt.tight_layout()
plt.show()

# --- All non-zero coefficients for reference ---
print("\nAll non-zero coefficients (sorted by magnitude):")
print(coef_series.reindex(coef_series.abs().sort_values(ascending=False).index)
      [coef_series.abs() > 0].to_string())